In [1]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))
STAGING = PROJECT_ROOT / "data" / "staging"

persons = pd.read_parquet(STAGING / "hf_persons_clean.parquet")
memberships = pd.read_parquet(STAGING / "hf_memberships_clean.parquet")
print(persons.shape, memberships.shape)

(45424, 7) (86234, 11)


In [2]:
herbert = persons[persons["last_key"].eq("BAUTISTA") & persons["first_key"].str.startswith("HERBERT")]
print(herbert[["id", "first_name", "last_name", "name_suffix"]].to_string())

cols = ["person_id", "year", "position", "province_std", "locality_std"]
memberships[memberships["person_id"].isin(herbert["id"])][cols].sort_values("year")

                              id           first_name last_name name_suffix
5623  01K7QAHFH1HNVC9H8ST4CTQ32W  Herbert Constantine  Bautista         NaN
5721  01K7QAHFGP4Z34Y2150ZJPQ5X5              Herbert  Bautista         NaN
6364  01K7QAHFGWTCT8N5V24GJ4G8VJ            Herbert C  Bautista         NaN


,person_id,year,position,province_std,locality_std
12128,01K7QAHFGWTCT8N5V24GJ4G8VJ,2004,VICE MAYOR,NCR SECOND DISTRICT,QUEZON
12129,01K7QAHFGWTCT8N5V24GJ4G8VJ,2007,VICE MAYOR,NCR SECOND DISTRICT,QUEZON
12130,01K7QAHFGWTCT8N5V24GJ4G8VJ,2010,MAYOR,NCR SECOND DISTRICT,QUEZON
10921,01K7QAHFGP4Z34Y2150ZJPQ5X5,2013,MAYOR,NCR SECOND DISTRICT,QUEZON
10738,01K7QAHFH1HNVC9H8ST4CTQ32W,2016,MAYOR,NCR SECOND DISTRICT,QUEZON


In [3]:
from src.transform.normalize import first_names_compatible

pp = memberships[["person_id", "province_std"]].drop_duplicates().merge(
    persons[["id", "first_name", "last_key", "name_suffix"]], left_on="person_id", right_on="id"
)
pp["suffix"] = pp["name_suffix"].fillna("")

pairs = pp.merge(pp, on=["province_std", "last_key", "suffix"], suffixes=("_a", "_b"))
pairs = pairs[pairs["person_id_a"] < pairs["person_id_b"]]
pairs = pairs[[first_names_compatible(a, b) for a, b in zip(pairs["first_name_a"], pairs["first_name_b"])]]

years = memberships.groupby("person_id")["year"].apply(set)
pairs["same_year"] = [bool(years[a] & years[b]) for a, b in zip(pairs["person_id_a"], pairs["person_id_b"])]

print("compatible pairs:", len(pairs))
print("never in office in the same year:", (~pairs["same_year"]).sum())
pairs.loc[~pairs["same_year"], ["province_std", "first_name_a", "first_name_b", "last_key"]].head(15)

compatible pairs: 1468
never in office in the same year: 1413


,province_std,first_name_a,first_name_b,last_key
32,CEBU,Lissa,Lissa Marie,STREEGAN
400,CEBU,Dean M,Dean Michael,SINGCO
440,MAGUINDANAO,Osmena,Osmeña,SINSUAT
458,PANGASINAN,Nicholi Jan,Nicholi Jan Louie,SISON
1091,SIQUIJOR,Remegio,Remegio Dele,SIERRAS
1268,COTABATO,Dyanne Kristine,Dyanne Kristine,SIDENO
1338,ISABELA,Celine,Celine Jeanne,SIQUIAN
1368,EASTERN SAMAR,Cornelio,Cornelio Ferdinand,SISON
1432,SARANGANI,Abdul,Abdul Montasher,SINGCOY
1613,ILOCOS SUR,John,John Patrick,SINGSON


In [5]:
from src.transform.normalize import name_key, name_tokens

winners = pd.read_parquet(STAGING / "openhalalan_winners_clean.parquet")
oh = winners[winners["year"].isin([2004, 2007, 2010, 2013, 2016]) & winners["middle_name"].notna()].copy()
oh["last_key"] = oh["last_name"].map(name_key, na_action="ignore")
oh["middle_key"] = oh["middle_name"].str.split().str[-1].map(name_key, na_action="ignore")

hf = memberships.merge(persons[["id", "first_name", "last_key"]], left_on="person_id", right_on="id", suffixes=("", "_p"))
keys = ["year", "position", "province_std", "last_key"]
cand = hf.merge(oh[keys + ["first_name", "middle_key"]], on=keys, suffixes=("", "_oh"))
cand = cand[[bool(name_tokens(a) & name_tokens(b)) for a, b in zip(cand["first_name"], cand["first_name_oh"])]]
cand = cand[cand.groupby("id")["middle_key"].transform("size") == 1]

person_mid = cand.groupby("person_id")["middle_key"].agg(lambda s: s.mode().iloc[0])
print("HF people with a middle name from OpenHalalan:", len(person_mid), "of", persons["id"].nunique())
print("Herbert Bautista's IDs:", person_mid.reindex(herbert["id"]).to_dict())

HF people with a middle name from OpenHalalan: 34665 of 45424
Herbert Bautista's IDs: {'01K7QAHFH1HNVC9H8ST4CTQ32W': nan, '01K7QAHFGP4Z34Y2150ZJPQ5X5': nan, '01K7QAHFGWTCT8N5V24GJ4G8VJ': 'MACLANG'}


In [6]:
pairs["mid_a"] = pairs["person_id_a"].map(person_mid)
pairs["mid_b"] = pairs["person_id_b"].map(person_mid)
both = pairs["mid_a"].notna() & pairs["mid_b"].notna()

pairs["evidence"] = "unknown"
pairs.loc[both & pairs["mid_a"].eq(pairs["mid_b"]), "evidence"] = "agree"
pairs.loc[both & pairs["mid_a"].ne(pairs["mid_b"]), "evidence"] = "conflict"
pairs.loc[pairs["same_year"], "evidence"] = "same_year"
print(pairs["evidence"].value_counts())

show = ["province_std", "first_name_a", "first_name_b", "last_key", "mid_a", "mid_b"]
print("\nCONFLICT examples:")
print(pairs.loc[pairs["evidence"].eq("conflict"), show].head(10).to_string())

evidence
unknown      929
agree        379
conflict     105
same_year     55
Name: count, dtype: int64

CONFLICT examples:
      province_std    first_name_a       first_name_b    last_key          mid_a    mid_b
3045       ANTIQUE             Pio     Pio Jessielito     SUMANDE                 CORDOVA
3058        ILOILO       Ernesto P      Ernesto Pablo  SUMAGAYSAY      GORRICETA    PABLO
3819         LEYTE  Marcos Antonio     Marcos Antonio      SOLANA        ANTONIO     DIWA
8542         ALBAY        Jose Ma.  Jose Ma. Clemente     SALCEDA             MA    SARTE
9078          CEBU          Gretel       Gretel Marie    BORGONIA        GEIRRAN  GIERRAN
9591          CEBU        Gretel M       Gretel Marie    BORGONIA        GEIRRAN  GIERRAN
9860          CEBU  Francis Sydney     Francis Sydney     BELCINA         SYDNEY  AMATONG
11508       IFUGAO          Isabel      Isabel Abella      BITTAG  ABELLACUCUHAN   ABELLA
12629         ABRA          Leo Pe             Leo Pe      BARONA  

In [7]:
from src.transform.normalize import middle_names_agree

first_words = persons.set_index("id")["first_name"].map(name_tokens)

def usable_mid(pid, mid):
    """A middle name counts only if it's not empty and not a word from the person's own first name."""
    return isinstance(mid, str) and mid != "" and mid not in first_words.get(pid, set())

pairs["mid_a_ok"] = [m if usable_mid(p, m) else None for p, m in zip(pairs["person_id_a"], pairs["mid_a"])]
pairs["mid_b_ok"] = [m if usable_mid(p, m) else None for p, m in zip(pairs["person_id_b"], pairs["mid_b"])]
both = pairs["mid_a_ok"].notna() & pairs["mid_b_ok"].notna()

agree = [both_ok and middle_names_agree(a, b) for both_ok, a, b in zip(both, pairs["mid_a_ok"], pairs["mid_b_ok"])]
pairs["evidence"] = "unknown"
pairs.loc[both, "evidence"] = "conflict"
pairs.loc[agree, "evidence"] = "agree"
pairs.loc[pairs["same_year"], "evidence"] = "same_year"
print(pairs["evidence"].value_counts())

show = ["province_std", "first_name_a", "first_name_b", "last_key", "mid_a_ok", "mid_b_ok"]
print("\nREMAINING CONFLICTS:")
print(pairs.loc[pairs["evidence"].eq("conflict"), show].head(10).to_string())

evidence
unknown      1000
agree         365
same_year      55
conflict       48
Name: count, dtype: int64

REMAINING CONFLICTS:
            province_std   first_name_a               first_name_b    last_key     mid_a_ok   mid_b_ok
12780              RIZAL           John               John Patrick    BAUTISTA        LOPEZ    MENDOZA
15483    NEGROS ORIENTAL           Mark           Mark Christoffel  BANQUERIGO  CHRISTOFFEL       LEON
16115             APAYAO           Mark                Mark Loreen    BAUTISTA      LACHICA     ARNEDO
17952      LANAO DEL SUR          Lumna          Lumna Abdul Bical  KURANGKING        ABDUL  VICEMAYOR
18161  NEGROS OCCIDENTAL           Mark         Mark Andrew Arthur       GOLEZ       ANDREW      JISON
18316      EASTERN SAMAR  Christopher S          Christopher Sheen    GONZALES    VICEMAYOR    PROJEMO
19414     SOUTHERN LEYTE         Rafael            Rafael Cromwell      GERONG       SERDAN        TAN
20585  NEGROS OCCIDENTAL        Matthew  Matthe

In [8]:
JOB_WORDS = {"MAYOR", "VICEMAYOR", "COUNCILOR", "GOVERNOR", "VICEGOVERNOR", "KAGAWAD", "BOARDMEMBER"}

def usable_pair_mid(mid, words):
    """A real middle name: longer than one letter, not a job title, not a word from either first name."""
    return isinstance(mid, str) and len(mid) > 1 and mid not in words and mid not in JOB_WORDS

words = [name_tokens(a) | name_tokens(b) for a, b in zip(pairs["first_name_a"], pairs["first_name_b"])]
pairs["mid_a_ok"] = [m if usable_pair_mid(m, w) else None for m, w in zip(pairs["mid_a"], words)]
pairs["mid_b_ok"] = [m if usable_pair_mid(m, w) else None for m, w in zip(pairs["mid_b"], words)]
both = pairs["mid_a_ok"].notna() & pairs["mid_b_ok"].notna()
agree = [ok and middle_names_agree(a, b) for ok, a, b in zip(both, pairs["mid_a_ok"], pairs["mid_b_ok"])]
pairs["evidence"] = "unknown"
pairs.loc[both, "evidence"] = "conflict"
pairs.loc[agree, "evidence"] = "agree"
pairs.loc[pairs["same_year"], "evidence"] = "same_year"
print(pairs["evidence"].value_counts(), "\n")

towns = memberships.dropna(subset=["locality_std"]).groupby("person_id")["locality_std"].apply(set)

def adjacent(ya, yb):
    """True if one career ends exactly one election (3 years) before the other starts."""
    early, late = (ya, yb) if max(ya) < min(yb) else (yb, ya)
    return min(late) - max(early) == 3

pairs["same_town"] = [bool(towns.get(a, set()) & towns.get(b, set())) for a, b in zip(pairs["person_id_a"], pairs["person_id_b"])]
pairs["adjacent"] = [adjacent(years[a], years[b]) for a, b in zip(pairs["person_id_a"], pairs["person_id_b"])]

known = pairs[pairs["evidence"].isin(["agree", "conflict"])]
print(pd.crosstab([known["same_town"], known["adjacent"]], known["evidence"], margins=True))
print()
print(pd.crosstab([pairs["same_town"], pairs["adjacent"]], pairs["evidence"]))

evidence
unknown      1022
agree         362
same_year      55
conflict       29
Name: count, dtype: int64 

evidence            agree  conflict  All
same_town adjacent                      
False     False        12         1   13
          True         29         2   31
True      False       106        13  119
          True        215        13  228
All                   362        29  391

evidence            agree  conflict  same_year  unknown
same_town adjacent                                     
False     False        12         1         21       67
          True         29         2          0      209
True      False       106        13         34      210
          True        215        13          0      536


In [10]:
accepted = pairs[pairs["evidence"].isin(["agree", "unknown"])]

parent = {pid: pid for pid in persons["id"]}

def find(x):
    """Follow the pointers from x to its group's representative."""
    while parent[x] != x:
        parent[x] = parent[parent[x]]
        x = parent[x]
    return x

for a, b in zip(accepted["person_id_a"], accepted["person_id_b"]):
    parent[find(a)] = find(b)

group = pd.Series({pid: find(pid) for pid in parent}, name="group")
sizes = group.value_counts()
multi = group[group.map(sizes) > 1]
print("IDs in merged groups:", len(multi), "| groups:", multi.nunique())
print("group sizes:", sizes[sizes > 1].value_counts().sort_index().to_dict())
print("Herbert's 3 IDs in one group:", group[herbert["id"]].nunique() == 1)

IDs in merged groups: 2633 | groups: 1288
group sizes: {2: 1236, 3: 47, 4: 5}
Herbert's 3 IDs in one group: True


In [11]:
def group_ok(members):
    """False if any two members were in office the same year, or have conflicting middle names."""
    yrs = [years.get(m, set()) for m in members]
    if sum(len(y) for y in yrs) != len(set().union(*yrs)):
        return False
    words = set().union(*(first_words.get(m, set()) for m in members))
    mids = [person_mid.get(m) for m in members]
    mids = [x for x in mids if usable_pair_mid(x, words)]
    return all(middle_names_agree(x, y) for i, x in enumerate(mids) for y in mids[i + 1:])

checks = multi.groupby(multi).apply(lambda s: group_ok(list(s.index)))
print("groups checked:", len(checks), "| failing the safeguard:", (~checks).sum())

groups checked: 1288 | failing the safeguard: 11


In [12]:
import importlib
import src.transform.clean_backfill as cb
importlib.reload(cb)

years_s = memberships.groupby("person_id")["year"].apply(set)
p2 = cb._split_id_pairs(persons, memberships, years_s)
mid2 = cb._hf_middle_names(persons, memberships, winners)
p2["evidence"] = cb._pair_evidence(p2, mid2)
print("middle names:", len(mid2))
print(p2["evidence"].value_counts().to_dict())

middle names: 34665
{'unknown': 1022, 'agree': 362, 'conflict': 29}


In [13]:
importlib.reload(cb)
merged, id_map = cb.resolve_persons(persons, memberships, winners)

print("people:", len(persons), "->", len(merged))
print(id_map["merge_evidence"].value_counts(dropna=False).to_dict())

uid = id_map.set_index("person_id_raw").loc[herbert["id"], "person_uid"].unique()
print(merged.loc[merged["id"].isin(uid), ["id", "first_name", "last_name", "n_ids", "merge_evidence", "name_variants"]].to_string())

people: 45424 -> 44106
{None: 42829, 'name_only': 1941, 'middle_name_agree': 654}
                              id           first_name last_name  n_ids merge_evidence                                                         name_variants
6207  01K7QAHFGWTCT8N5V24GJ4G8VJ  Herbert Constantine  Bautista      3      name_only  Herbert Bautista | Herbert C Bautista | Herbert Constantine Bautista


### Summary: merging split person IDs
- Problem: HF gives one person a new ID when the name is written differently in a later election (e.g. Herbert Bautista: "Herbert C" 2004–2010, "Herbert" 2013, "Herbert Constantine" 2016). This inflates dynasty measures and the dashboard network.
- Candidate pairs: same province, surname key, and suffix, compatible first names, never in office the same year: 1,413 pairs.
- OpenHalalan middle names (34,665 HF people): 362 agree, 29 conflict, 1,022 unknown. Conflict rate among pairs with evidence: 7.4%, and not lower for same-town or back-to-back pairs.
- Decision (Option B): merge agree + unknown, label each merge (`middle_name_agree` / `name_only`), never merge conflicts or same-year pairs. Groups failing the safeguard (11) are not merged.
- Result: 45,424 → 44,106 people. Implemented in `resolve_persons()` in `clean_backfill.py`. All 6 data quality checks pass.